# 💳 Credit Card Fraud Detection & Transaction Analytics
### SQL + Python Project — Step by Step

**How this project works:**
- 🐍 **Python** is used ONLY to generate/clean messy raw data and load it into MySQL
- 🗄️ **MySQL (SQL)** does ALL the analytics and fraud detection

Run each cell below in order (Shift + Enter). Make sure MySQL/MariaDB is running and you've created the database + user (see README Step 1) before running the DB cells.


## Step 1: Import Libraries & Set Database Config

In [ ]:
import re
import pandas as pd
import numpy as np
from sqlalchemy import create_engine, text

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 150)


In [ ]:
# ---- MySQL connection settings (edit if your credentials are different) ----
DB_USER = "fintech_user"
DB_PASSWORD = "fintech_pass123"
DB_HOST = "localhost"
DB_PORT = 3306
DB_NAME = "fintech_db"

engine = create_engine(f"mysql+pymysql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}")
print("Database engine created. (This doesn't connect yet — connection happens on first query.)")


## Step 2: Generate Messy Raw Data
This simulates a real bank export — with duplicates, missing values, inconsistent formatting, and bad amounts.
If you already have `data_raw/*.csv` from before, you can skip this cell.


In [ ]:
import random
from datetime import datetime, timedelta
from faker import Faker

fake = Faker()
Faker.seed(42)
random.seed(42)

N_CUSTOMERS = 500
N_MERCHANTS = 60
N_TRANSACTIONS = 8500

CATEGORIES = ["Groceries", "Utilities", "Electronics", "Travel", "Dining",
              "Entertainment", "Healthcare", "Education", "Fuel", "Subscriptions",
              "ATM Withdrawal", "P2P Transfer"]
PAYMENT_METHODS = ["UPI", "Credit Card", "Debit Card", "Net Banking", "Wallet"]
CITIES = ["Mumbai", "Delhi", "Bengaluru", "Raipur", "Pune", "Hyderabad",
          "Chennai", "Kolkata", "Ahmedabad", "Jaipur"]

def messy_case(text):
    r = random.random()
    if r < 0.25:
        return text.upper()
    elif r < 0.5:
        return text.lower()
    elif r < 0.65:
        return f"  {text}  "
    return text

print("Setup done. Categories, cities, and payment methods defined.")


In [ ]:
# Generate raw customers (with some missing values + duplicates)
customers_raw = []
for cid in range(1, N_CUSTOMERS + 1):
    signup = fake.date_between(start_date="-3y", end_date="-30d")
    age = random.randint(18, 65)
    if random.random() < 0.04:
        age = ""
    credit_score = random.randint(300, 900)
    if random.random() < 0.03:
        credit_score = ""
    customers_raw.append({
        "customer_id": cid, "name": fake.name(), "age": age,
        "city": messy_case(random.choice(CITIES)),
        "account_type": messy_case(random.choice(["Savings", "Current", "Salary"])),
        "signup_date": signup.isoformat(), "credit_score": credit_score
    })

# inject duplicate rows
customers_raw.extend(random.sample(customers_raw, 15))

customers_raw_df = pd.DataFrame(customers_raw)
print(f"Generated {len(customers_raw_df)} raw customer rows (includes 15 duplicates)")
customers_raw_df.head()


In [ ]:
# Generate raw merchants
merchants_raw = []
for mid in range(1, N_MERCHANTS + 1):
    merchants_raw.append({
        "merchant_id": mid, "merchant_name": fake.company(),
        "category": messy_case(random.choice(CATEGORIES)),
        "city": messy_case(random.choice(CITIES))
    })

merchants_raw_df = pd.DataFrame(merchants_raw)
print(f"Generated {len(merchants_raw_df)} raw merchant rows")
merchants_raw_df.head()


In [ ]:
# Generate raw transactions (with dirty amounts, mixed date formats, duplicates)
base_amounts = {
    "Groceries": (200, 3000), "Utilities": (300, 5000), "Electronics": (1000, 80000),
    "Travel": (500, 60000), "Dining": (150, 4000), "Entertainment": (100, 3000),
    "Healthcare": (200, 20000), "Education": (500, 50000), "Fuel": (200, 3000),
    "Subscriptions": (99, 1500), "ATM Withdrawal": (500, 20000), "P2P Transfer": (50, 50000),
}

def random_date_format(dt):
    fmt = random.choice(["%Y-%m-%d %H:%M:%S", "%d/%m/%Y %H:%M", "%d-%m-%Y %H:%M:%S", "%Y/%m/%d %H:%M"])
    return dt.strftime(fmt)

start_date = datetime.now() - timedelta(days=180)
transactions_raw = []
for tid in range(1, N_TRANSACTIONS + 1):
    cust = random.choice(customers_raw)
    merch = random.choice(merchants_raw)
    category_clean = merch["category"].strip().title()
    low, high = base_amounts.get(category_clean, (100, 5000))
    amount = round(random.uniform(low, high), 2)

    r = random.random()
    if r < 0.03:
        amount_str = f"\u20b9{amount:,.2f}"
    elif r < 0.05:
        amount_str = str(-amount)
    elif r < 0.06:
        amount_str = ""
    elif r < 0.07:
        amount_str = "N/A"
    else:
        amount_str = str(amount)

    txn_time = start_date + timedelta(days=random.randint(0, 180), hours=random.randint(0, 23), minutes=random.randint(0, 59))

    transactions_raw.append({
        "transaction_id": tid, "customer_id": cust["customer_id"], "merchant_id": merch["merchant_id"],
        "amount": amount_str, "category": messy_case(category_clean),
        "payment_method": messy_case(random.choice(PAYMENT_METHODS)),
        "txn_datetime": random_date_format(txn_time),
        "txn_city": messy_case(random.choice(CITIES))
    })

transactions_raw.extend(random.sample(transactions_raw, int(N_TRANSACTIONS * 0.02)))
random.shuffle(transactions_raw)

transactions_raw_df = pd.DataFrame(transactions_raw)
print(f"Generated {len(transactions_raw_df)} raw transaction rows (with dirty amounts, mixed dates, duplicates)")
transactions_raw_df.head(10)


In [ ]:
import os
os.makedirs("data_raw", exist_ok=True)
customers_raw_df.to_csv("data_raw/customers_raw.csv", index=False)
merchants_raw_df.to_csv("data_raw/merchants_raw.csv", index=False)
transactions_raw_df.to_csv("data_raw/transactions_raw.csv", index=False)
print("Raw CSVs saved to data_raw/")


## Step 3: Data Cleaning Functions
Python's ONLY job in this project — cleaning the messy data before it goes into MySQL.


In [ ]:
def clean_text(series):
    """Strip whitespace and standardize to Title Case."""
    return series.astype(str).str.strip().str.title()

def clean_amount(value):
    """Remove currency symbols/commas, coerce to float. Returns None if invalid."""
    if pd.isna(value):
        return None
    s = str(value).strip()
    s = re.sub(r"[\u20b9$,]", "", s)
    try:
        val = float(s)
        return val if val > 0 else None
    except ValueError:
        return None

def parse_mixed_dates(series):
    """Try multiple known formats, fall back to pandas' flexible parser."""
    known_formats = ["%Y-%m-%d %H:%M:%S", "%d/%m/%Y %H:%M", "%d-%m-%Y %H:%M:%S", "%Y/%m/%d %H:%M"]
    result = pd.Series([pd.NaT] * len(series), index=series.index)
    remaining = series.copy()
    for fmt in known_formats:
        mask = result.isna()
        parsed = pd.to_datetime(remaining[mask], format=fmt, errors="coerce")
        result.loc[mask] = parsed
    mask = result.isna()
    if mask.any():
        result.loc[mask] = pd.to_datetime(remaining[mask], errors="coerce")
    return result

print("Cleaning functions defined: clean_text(), clean_amount(), parse_mixed_dates()")


## Step 4: Clean the Customers Data

In [ ]:
df_customers = pd.read_csv("data_raw/customers_raw.csv")
before = len(df_customers)

df_customers = df_customers.drop_duplicates(subset="customer_id")
df_customers["name"] = df_customers["name"].astype(str).str.strip()
df_customers["city"] = clean_text(df_customers["city"])
df_customers["account_type"] = clean_text(df_customers["account_type"])
df_customers["signup_date"] = pd.to_datetime(df_customers["signup_date"], errors="coerce")

df_customers["age"] = pd.to_numeric(df_customers["age"], errors="coerce")
df_customers["age"] = df_customers["age"].fillna(df_customers["age"].median()).astype(int)

df_customers["credit_score"] = pd.to_numeric(df_customers["credit_score"], errors="coerce")
df_customers["credit_score"] = df_customers["credit_score"].fillna(df_customers["credit_score"].median()).astype(int)

df_customers = df_customers.dropna(subset=["signup_date"])

print(f"Customers: {before} raw rows -> {len(df_customers)} clean rows")
df_customers.head()


## Step 5: Clean the Merchants Data

In [ ]:
df_merchants = pd.read_csv("data_raw/merchants_raw.csv")
before = len(df_merchants)

df_merchants = df_merchants.drop_duplicates(subset="merchant_id")
df_merchants["merchant_name"] = df_merchants["merchant_name"].astype(str).str.strip()
df_merchants["category"] = clean_text(df_merchants["category"])
df_merchants["city"] = clean_text(df_merchants["city"])

print(f"Merchants: {before} raw rows -> {len(df_merchants)} clean rows")
df_merchants.head()


## Step 6: Clean the Transactions Data

In [ ]:
df_transactions = pd.read_csv("data_raw/transactions_raw.csv")
before = len(df_transactions)

df_transactions = df_transactions.drop_duplicates(subset="transaction_id")

df_transactions["amount"] = df_transactions["amount"].apply(clean_amount)
df_transactions["category"] = clean_text(df_transactions["category"])
df_transactions["payment_method"] = clean_text(df_transactions["payment_method"])
df_transactions["txn_city"] = clean_text(df_transactions["txn_city"])
df_transactions["txn_datetime"] = parse_mixed_dates(df_transactions["txn_datetime"])

n_before_dropna = len(df_transactions)
df_transactions = df_transactions.dropna(subset=["amount", "txn_datetime", "customer_id", "merchant_id"])
n_invalid = n_before_dropna - len(df_transactions)

print(f"Transactions: {before} raw rows -> {len(df_transactions)} clean rows")
print(f"  ({before - n_before_dropna} exact duplicates removed, {n_invalid} invalid/missing rows dropped)")
df_transactions.head()


## Step 7: Load Clean Data into MySQL
Make sure MySQL is running and `fintech_db` + `fintech_user` exist (see README Step 1).
This applies `sql/schema.sql` first (creates tables), then loads the cleaned DataFrames.


In [ ]:
with open("sql/schema.sql") as f:
    schema_sql = f.read()

statements = [s.strip() for s in schema_sql.split(";") if s.strip()]
with engine.connect() as conn:
    for stmt in statements:
        conn.execute(text(stmt))
    conn.commit()

print("Schema applied — tables created in MySQL.")


In [ ]:
df_customers.to_sql("customers", engine, if_exists="append", index=False)
df_merchants.to_sql("merchants", engine, if_exists="append", index=False)
df_transactions.to_sql("transactions", engine, if_exists="append", index=False)

print("Clean data loaded into MySQL tables: customers, merchants, transactions")


## Step 8: Run SQL Analytics
From here on, **all logic lives in SQL** — Python is just fetching and displaying the results.


In [ ]:
query_category_totals = """
SELECT category,
       COUNT(*) AS total_txns,
       ROUND(SUM(amount), 2) AS total_value,
       ROUND(AVG(amount), 2) AS avg_txn_value
FROM transactions
GROUP BY category
ORDER BY total_value DESC;
"""

df_result = pd.read_sql(text(query_category_totals), engine)
df_result


In [ ]:
query_monthly_trend = """
SELECT DATE_FORMAT(txn_datetime, '%Y-%m') AS month,
       COUNT(*) AS txn_count,
       ROUND(SUM(amount), 2) AS total_value
FROM transactions
GROUP BY month
ORDER BY month;
"""

df_monthly = pd.read_sql(text(query_monthly_trend), engine)
df_monthly


In [ ]:
query_top_customers = """
SELECT c.customer_id, c.name, c.city,
       COUNT(t.transaction_id) AS num_txns,
       ROUND(SUM(t.amount), 2) AS total_spent
FROM transactions t
JOIN customers c ON c.customer_id = t.customer_id
GROUP BY c.customer_id, c.name, c.city
ORDER BY total_spent DESC
LIMIT 10;
"""

df_top_customers = pd.read_sql(text(query_top_customers), engine)
df_top_customers


In [ ]:
query_city_rank = """
SELECT customer_id, name, city, total_spent,
       RANK() OVER (PARTITION BY city ORDER BY total_spent DESC) AS city_rank
FROM (
    SELECT c.customer_id, c.name, c.city, ROUND(SUM(t.amount), 2) AS total_spent
    FROM transactions t
    JOIN customers c ON c.customer_id = t.customer_id
    GROUP BY c.customer_id, c.name, c.city
) spend_by_customer
ORDER BY city, city_rank
LIMIT 20;
"""

df_city_rank = pd.read_sql(text(query_city_rank), engine)
df_city_rank


## Step 9: Fraud Detection — Pure SQL Rule Engine
A transaction is scored on 4 rules (each worth 1 point):
1. Amount is a statistical outlier for its category (mean + 2×stddev)
2. Odd-hour transaction (12 AM–5 AM)
3. Transaction city ≠ customer's home city
4. Large P2P/ATM transaction via a risky payment channel

Score ≥ 3 → flagged as high-risk fraud.


In [ ]:
with open("sql/analysis_and_fraud.sql") as f:
    fraud_sql = f.read()

# Extract just the fraud-scoring query (Section B1) to run directly here
fraud_query = """
WITH category_stats AS (
    SELECT category, AVG(amount) AS avg_amount, STDDEV(amount) AS stddev_amount
    FROM transactions GROUP BY category
),
scored_transactions AS (
    SELECT
        t.transaction_id, t.customer_id, t.merchant_id, t.amount, t.category,
        t.payment_method, t.txn_datetime, t.txn_city, c.city AS home_city,
        CASE WHEN t.amount > (cs.avg_amount + 2 * cs.stddev_amount) THEN 1 ELSE 0 END AS rule_outlier_amount,
        CASE WHEN HOUR(t.txn_datetime) BETWEEN 0 AND 5 THEN 1 ELSE 0 END AS rule_odd_hour,
        CASE WHEN t.txn_city <> c.city THEN 1 ELSE 0 END AS rule_city_mismatch,
        CASE WHEN t.category IN ('P2P Transfer', 'Atm Withdrawal')
                  AND t.payment_method IN ('Wallet', 'Net Banking')
                  AND t.amount > 15000
             THEN 1 ELSE 0 END AS rule_highrisk_channel
    FROM transactions t
    JOIN customers c ON c.customer_id = t.customer_id
    JOIN category_stats cs ON cs.category = t.category
)
SELECT *,
       (rule_outlier_amount + rule_odd_hour + rule_city_mismatch + rule_highrisk_channel) AS fraud_score,
       CASE WHEN (rule_outlier_amount + rule_odd_hour + rule_city_mismatch + rule_highrisk_channel) >= 3
            THEN 1 ELSE 0 END AS is_flagged_fraud
FROM scored_transactions
ORDER BY fraud_score DESC, amount DESC;
"""

df_fraud = pd.read_sql(text(fraud_query), engine)
print(f"Total transactions scored: {len(df_fraud)}")
print(f"Flagged as high-risk fraud: {df_fraud['is_flagged_fraud'].sum()}")
df_fraud[df_fraud['is_flagged_fraud'] == 1].head(15)


In [ ]:
query_fraud_by_category = """
WITH category_stats AS (
    SELECT category, AVG(amount) AS avg_amount, STDDEV(amount) AS stddev_amount
    FROM transactions GROUP BY category
),
flagged AS (
    SELECT t.transaction_id, t.category,
        (
          CASE WHEN t.amount > (cs.avg_amount + 2*cs.stddev_amount) THEN 1 ELSE 0 END +
          CASE WHEN HOUR(t.txn_datetime) BETWEEN 0 AND 5 THEN 1 ELSE 0 END +
          CASE WHEN t.txn_city <> c.city THEN 1 ELSE 0 END +
          CASE WHEN t.category IN ('P2P Transfer','Atm Withdrawal')
                    AND t.payment_method IN ('Wallet','Net Banking')
                    AND t.amount > 15000 THEN 1 ELSE 0 END
        ) AS fraud_score
    FROM transactions t
    JOIN customers c ON c.customer_id = t.customer_id
    JOIN category_stats cs ON cs.category = t.category
)
SELECT category,
       COUNT(*) AS total_txns,
       SUM(CASE WHEN fraud_score >= 3 THEN 1 ELSE 0 END) AS flagged_fraud_txns,
       ROUND(100.0 * SUM(CASE WHEN fraud_score >= 3 THEN 1 ELSE 0 END) / COUNT(*), 2) AS fraud_rate_pct
FROM flagged
GROUP BY category
ORDER BY fraud_rate_pct DESC;
"""

df_fraud_by_cat = pd.read_sql(text(query_fraud_by_category), engine)
df_fraud_by_cat


## Step 10 (Optional): Quick Visual Summary
Just a simple chart to *see* your results — this is optional and separate from the SQL analysis above.


In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8,5))
plt.barh(df_fraud_by_cat["category"], df_fraud_by_cat["fraud_rate_pct"], color="crimson")
plt.xlabel("Fraud Rate (%)")
plt.title("Fraud Rate by Category")
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()


## ✅ Done!
You've now:
1. Generated messy raw data
2. Cleaned it with Python
3. Loaded it into MySQL
4. Run all analytics and fraud detection **in SQL**

This notebook itself is a great thing to show in your GitHub repo — add it as `Credit_Card_Fraud_Detection_Analysis.ipynb` alongside your other files.
